# Lesson 6.4: z-Tests for Means and Proportions

Run a z-test for a mean (the 50 bolts) and a z-test for a proportion (184 of 200 customers satisfied), by hand and with statsmodels.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. A z-test for a mean, by hand

A line makes 10 mm bolts with a known SD of 0.5 mm. A sample of 50 bolts averages 10.14 mm. The z statistic is the gap divided by the standard error, and the two-sided p-value comes from the normal curve.

In [1]:
import numpy as np
from scipy import stats

mu0, sigma, n, xbar = 10, 0.5, 50, 10.14

se = sigma / np.sqrt(n)
z = (xbar - mu0) / se
p = 2 * stats.norm.sf(abs(z))
ci = (xbar - 1.96 * se, xbar + 1.96 * se)

print(f'z = {z:.3f}, p = {p:.4f}')
print(f'95% CI: ({ci[0]:.3f}, {ci[1]:.3f})')

z = 1.980, p = 0.0477
95% CI: (10.001, 10.279)


## 2. The same test with ztest()

statsmodels needs raw data. We build 50 pretend bolt readings with mean 10.14 and SD 0.5, so the result matches the hand calculation. Note that ztest() estimates the SD from the data rather than using a known sigma.

In [2]:
from statsmodels.stats.weightstats import ztest

rng = np.random.default_rng(42)
x = rng.normal(size=50)
bolts = 10.14 + 0.5 * (x - x.mean()) / x.std(ddof=1)   # mean 10.14, SD 0.5

z_stat, p_val = ztest(bolts, value=10, alternative='two-sided')
print(f'mean = {bolts.mean():.2f}, sd = {bolts.std(ddof=1):.2f}')
print(f'z = {z_stat:.3f}, p = {p_val:.4f}')

mean = 10.14, sd = 0.50
z = 1.980, p = 0.0477


## 3. A z-test for a proportion

A company claims 95% satisfaction, but 184 of 200 customers are satisfied. The claim says 'at least 95%', so we test the left tail ('smaller'). We set prop_var=0.95 so the standard error uses the claimed value, as a test should.

In [3]:
from statsmodels.stats.proportion import proportions_ztest

z_prop, p_prop = proportions_ztest(count=184, nobs=200, value=0.95,
                                   alternative='smaller', prop_var=0.95)

print(f'sample proportion = {184 / 200:.2f}')
print(f'z = {z_prop:.3f}')
print(f'left-tailed p = {p_prop:.4f}')
print('reject H0 at 0.05?', p_prop <= 0.05)

sample proportion = 0.92
z = -1.947
left-tailed p = 0.0258
reject H0 at 0.05? True


## 4. Why the standard error matters

If we leave out prop_var, statsmodels uses the observed 0.92 in the standard error. This is the classic mistake from the lesson, and it changes z and p.

In [4]:
z_wrong, p_wrong = proportions_ztest(count=184, nobs=200, value=0.95,
                                     alternative='smaller')

print(f'with the observed proportion: z = {z_wrong:.3f}, p = {p_wrong:.4f}')
print(f'with the claimed 0.95:         z = {z_prop:.3f}, p = {p_prop:.4f}')

with the observed proportion: z = -1.564, p = 0.0589
with the claimed 0.95:         z = -1.947, p = 0.0258


## Try this

- Change the sample mean to 10.05 and predict whether the p-value gets bigger or smaller, then run it.
- Change the alternative from 'smaller' to 'two-sided' in the proportion test and predict what happens to p.

## The same idea in R, SPSS and Stata

**R**

```r
# z-test for a mean (sigma known), by hand: base R has no z.test
z <- (10.14 - 10) / (0.5 / sqrt(50))
z
2 * pnorm(-abs(z))

# z-test for a proportion, no continuity correction
res <- prop.test(184, 200, p = 0.95, alternative = "less", correct = FALSE)
res
-sqrt(res$statistic)   # the z value (z squared is X-squared)
```

**SPSS** (Transform > Compute Variable (SPSS has no z-test menu; use the normal-curve functions))

```
DATA LIST FREE / xbar mu0 sigma n x m p0.
BEGIN DATA
10.14 10 0.5 50 184 200 0.95
END DATA.
COMPUTE z_mean = (xbar - mu0) / (sigma / SQRT(n)).
COMPUTE p_mean = 2 * (1 - CDF.NORMAL(ABS(z_mean), 0, 1)).
COMPUTE z_prop = (x / m - p0) / SQRT(p0 * (1 - p0) / m).
COMPUTE p_prop = CDF.NORMAL(z_prop, 0, 1).
LIST z_mean p_mean z_prop p_prop.
```

**Stata**

```
display (10.14 - 10) / (0.5 / sqrt(50))
display 2 * (1 - normal(1.9799))
prtesti 200 184 0.95, count
```